In [0]:
%run "./03_conexao_adls_rafael"

In [0]:
# Célula 2 — regras que faltam para ecommerce_produtos
# (técnica 1-4 e negócio nenhuma já cobertas pelo notebook 3; aqui só o que falta)

def regras_extras_produtos(df_batch, df_categorias, batch_id, ref_date, tabela, agora):
    total = df_batch.count()
    linhas = []

    # técnica 5: unidade_medida em domínio válido
    validos = ["un", "kg", "g", "L", "ml"]
    n = df_batch.filter(~F.col("unidade_medida").isin(validos)).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "unidade_medida", "invalid_domain_pct", total, n, "medium", agora))

    # negócio 6: is_ativo não nulo (booleano já garantido pelo schema de leitura)
    n = df_batch.filter(F.col("is_ativo").isNull()).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "is_ativo", "null_percentage", total, n, "high", agora))

    # negócio 7: preco_lista <= 5000
    n = df_batch.filter(F.col("preco_lista") > 5000).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "preco_lista", "above_max_price_pct", total, n, "medium", agora))

    # negócio 8: nome_marca não nulo/vazio
    n = df_batch.filter(F.col("nome_marca").isNull() | (F.trim(F.col("nome_marca")) == "")).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "nome_marca", "null_or_empty_pct", total, n, "medium", agora))

    # negócio 9: BLOQUEADA — depende de ecommerce_itens_pedido/ecommerce_pedidos (outra dupla)
    linhas.append((batch_id, ref_date, tabela, "sku", "active_no_sale_90d_pct",
                    None, total, None, None, "high", "blocked", agora))

    # negócio 10: cada SKU deve pertencer a subcategoria (categoria com pai preenchido)
    subcategorias = df_categorias.filter(F.col("id_categoria_pai").isNotNull()).select("id_categoria")
    n = df_batch.join(subcategorias, "id_categoria", "left_anti").count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "not_subcategory_pct", total, n, "medium", agora))

    return linhas


linhas_produtos_extras = []
for b in sorted(r["_batch_id"] for r in df_produtos.select("_batch_id").distinct().collect()):
    sub = df_produtos.filter(F.col("_batch_id") == b)
    ref_date = sub.select("_reference_date").first()["_reference_date"]
    linhas_produtos_extras += regras_extras_produtos(sub, df_categorias, b, ref_date, "ecommerce_produtos", agora)

In [0]:
# Célula 3 — as 10 regras de ecommerce_categorias (nenhuma feita ainda)

def regras_categorias(df_cat, df_produtos_ref, agora):
    total = df_cat.count()
    batch_id, ref_date, tabela = "dimensao", None, "ecommerce_categorias"
    linhas = []

    # técnica 1: id_categoria não nulo nem duplicado (PK)
    n_nulo = df_cat.filter(F.col("id_categoria").isNull()).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "null_percentage", total, n_nulo, "high", agora))
    n_dup = total - df_cat.dropDuplicates(["id_categoria"]).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "duplicate_percentage", total, n_dup, "high", agora))

    # técnica 2: nome_categoria não nulo/vazio
    n = df_cat.filter(F.col("nome_categoria").isNull() | (F.trim(F.col("nome_categoria")) == "")).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "nome_categoria", "null_or_empty_pct", total, n, "high", agora))

    # técnica 3: id_categoria_pai, quando preenchido, deve existir em id_categoria
    n = (df_cat.filter(F.col("id_categoria_pai").isNotNull())
         .join(df_cat.select(F.col("id_categoria").alias("id_categoria_pai")), "id_categoria_pai", "left_anti")
         .count())
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria_pai", "orphan_pai_pct", total, n, "high", agora))

    # técnica 4: sem ciclo direto (categoria não pode ser pai de si mesma)
    n = df_cat.filter(F.col("id_categoria") == F.col("id_categoria_pai")).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria_pai", "self_reference_pct", total, n, "high", agora))

    # técnica 5: nome_categoria único dentro do mesmo pai (raízes agrupadas juntas)
    dup = (df_cat.groupBy(F.coalesce(F.col("id_categoria_pai"), F.lit(-1)).alias("pai_grp"), "nome_categoria")
           .count().filter("count > 1"))
    n = dup.agg(F.sum("count")).first()[0] or 0
    linhas.append(_linha(batch_id, ref_date, tabela, "nome_categoria", "duplicate_name_in_parent_pct", total, int(n), "medium", agora))

    # negócio 6: hierarquia com exatamente 2 níveis (pai de subcategoria não pode ter pai também)
    subcats = df_cat.filter(F.col("id_categoria_pai").isNotNull())
    avos = (subcats.join(df_cat.select(F.col("id_categoria").alias("id_categoria_pai"),
                                        F.col("id_categoria_pai").alias("avo")),
                          "id_categoria_pai")
            .filter(F.col("avo").isNotNull()))
    n = avos.count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria_pai", "hierarchy_depth_gt2_pct", total, n, "high", agora))

    # negócio 7: toda subcategoria deve ter >= 1 produto associado
    subcats_ids = subcats.select("id_categoria")
    total_subcats = subcats_ids.count()
    produtos_categorias = df_produtos_ref.select("id_categoria").distinct()
    n = subcats_ids.join(produtos_categorias, "id_categoria", "left_anti").count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "subcategory_without_product_pct", total_subcats, n, "medium", agora))

    # negócio 8: toda categoria raiz deve ter >= 1 subcategoria
    raizes = df_cat.filter(F.col("id_categoria_pai").isNull()).select(F.col("id_categoria").alias("id_categoria_pai")).distinct()
    total_raizes = raizes.count()
    raizes_com_sub = subcats.select("id_categoria_pai").distinct()
    n = raizes.join(raizes_com_sub, "id_categoria_pai", "left_anti").count()
    linhas.append(_linha(batch_id, ref_date, tabela, "id_categoria", "root_without_subcategory_pct", total_raizes, n, "medium", agora))

    # negócio 9: nome_categoria sem caracteres inválidos
    n = df_cat.filter(F.col("nome_categoria").rlike(r"[<>&]|(?i)script")).count()
    linhas.append(_linha(batch_id, ref_date, tabela, "nome_categoria", "invalid_characters_pct", total, n, "high", agora))

    # negócio 10: BLOQUEADA — número fixo de categorias raiz em config.py é desconhecido
    linhas.append((batch_id, ref_date, tabela, "id_categoria", "root_count_matches_config_pct",
                    None, total, int(total_raizes), None, "medium", "blocked", agora))

    return linhas


ultimo_batch = sorted(r["_batch_id"] for r in df_produtos.select("_batch_id").distinct().collect())[-1]
df_produtos_ref = df_produtos.filter(F.col("_batch_id") == ultimo_batch)
linhas_categorias = regras_categorias(df_categorias, df_produtos_ref, agora)

In [0]:
# Célula 4 — junta tudo e grava
df_novas_metricas = spark.createDataFrame(linhas_produtos_extras + linhas_categorias, schema=SCHEMA_METRICAS)
df_novas_metricas.orderBy("status", "batch_id").show(50, truncate=False)

gravar_append_sem_duplicar(df_novas_metricas, TABELA_METRICAS,
                            ["batch_id", "table_name", "column_name", "metric_name"])